# Decision Trees, Random Forests, And Small Ensembles

**Purpose:** show how tree models, random forests, majority voting, and stacking behave on a compact multiclass classification problem.

**Dataset:** Iris flowers from `sklearn.datasets.load_iris()` with four numeric measurements and three species.

**Method:** select a tree-based model with train-only cross-validation, evaluate once on a holdout split, then use voting and stacking as educational ensemble demonstrations.

**Metric:** accuracy and macro F1 because the Iris classes are balanced.

**Headline takeaway:** the selected random forest reaches holdout macro F1 `0.949`; the stacking section uses out-of-fold base predictions to avoid training its meta-model on in-sample outputs.


## Imports And Data Load

Iris is a small teaching benchmark. The value of this notebook is not dataset novelty; it is the ensemble mechanics.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from ml_portfolio.plotting import ACCENT, CAPTION, HIGHLIGHT, MUTED, apply_portfolio_style, save_figure
from scipy.stats import mode
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists() and (PROJECT_ROOT.parent / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

apply_portfolio_style()

RANDOM_STATE = 42
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Rows: {X.shape[0]} | Features: {X.shape[1]} | Classes: {len(target_names)}")


## Create A Stratified Holdout Split

The test split is shared by the later educational demos, so only the first selected-model evaluation should be read as the primary benchmark.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {X_train.shape[0]} | Test rows: {X_test.shape[0]}")


## Cross-Validate Tree Depth And Model Family

Single trees are easy to inspect; forests reduce variance. Both are compared over the same depth values using only the training split.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
depths = [1, 2, 3, 4, 5, None]
rows = []

for depth in depths:
    candidates = {
        "Decision Tree": DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            max_depth=depth,
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
    }
    for model_name, model in candidates.items():
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
        rows.append(
            {
                "model": model_name,
                "max_depth": "None" if depth is None else depth,
                "cv_accuracy_mean": scores.mean(),
                "cv_accuracy_std": scores.std(),
            }
        )


## Select The Primary Model

Tie-breaking favors a single decision tree when scores are effectively identical, because simpler models are easier to explain.


In [ ]:
cv_results = pd.DataFrame(rows)
model_rank = {"Decision Tree": 0, "Random Forest": 1}
cv_results["tie_rank"] = cv_results["model"].map(model_rank)
cv_results = cv_results.sort_values(
    ["cv_accuracy_mean", "cv_accuracy_std", "tie_rank"],
    ascending=[False, True, True],
)
display(cv_results.drop(columns="tie_rank"))

best = cv_results.iloc[0]
best_depth = None if best["max_depth"] == "None" else int(best["max_depth"])
if best["model"] == "Decision Tree":
    selected_model = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE)
else:
    selected_model = RandomForestClassifier(
        max_depth=best_depth,
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

print(f"Selected model: {best['model']} with max_depth={best['max_depth']}")


## Final Holdout Metrics

This is the primary benchmark for the notebook. Later ensemble sections reuse the split as demonstrations.


In [ ]:
selected_model.fit(X_train, y_train)
y_pred = selected_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Test accuracy: {accuracy:.3f}")
print(f"Test macro F1: {macro_f1:.3f}")
print()
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=target_names))


## Confusion Matrix And Feature Importances

The confusion matrix shows class-level errors. Feature importances are a quick tree diagnostic, not a causal explanation.


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=target_names, cmap="Blues")
plt.title("Selected tree model confusion matrix")
plt.tight_layout()
plt.show()

importances = getattr(selected_model, "feature_importances_", None)
if importances is not None:
    importance_table = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values(
        "importance",
        ascending=False,
    )
    display(importance_table)

    plot_importance = importance_table.sort_values("importance")
    fig, ax = plt.subplots()
    colors = [HIGHLIGHT if feature == plot_importance["feature"].iloc[-1] else ACCENT for feature in plot_importance["feature"]]
    bars = ax.barh(plot_importance["feature"], plot_importance["importance"], color=colors)
    ax.bar_label(bars, labels=[f"{value:.3f}" for value in plot_importance["importance"]], padding=3, fontsize=9)
    ax.set_xlabel("Impurity-based importance")
    ax.set_ylabel("Feature")
    ax.set_title("Petal measurements drive the selected Iris forest")
    ax.text(
        0,
        -0.22,
        "Iris holdout; importances summarize the fitted random forest, not causal biology.",
        transform=ax.transAxes,
        color=CAPTION,
        fontsize=9,
    )
    fig.tight_layout()
    save_figure(fig, "iris_random_forest_importances", project_root=PROJECT_ROOT)
    plt.show()

## Train A Small Tree For Manual Path Inspection

A shallow decision tree makes split logic visible row by row.


In [ ]:
path_tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
path_tree.fit(X_train, y_train)

plt.figure(figsize=(12, 6))
plot_tree(path_tree, feature_names=feature_names, class_names=target_names, filled=True)
plt.title("Decision tree used for manual path inspection")
plt.show()


## Walk One Prediction By Hand

This cell follows the first test row from the root node to the terminal leaf.


In [ ]:
first_row = X_test[0]
node_id = 0
print("First test row:")
for name, value in zip(feature_names, first_row):
    print(f"{name}: {value:.3f}")

print()
print("Manual decision path:")
tree_ = path_tree.tree_
while tree_.children_left[node_id] != tree_.children_right[node_id]:
    feature_idx = tree_.feature[node_id]
    threshold = tree_.threshold[node_id]
    value = first_row[feature_idx]
    go_left = value <= threshold
    print(
        f"Node {node_id}: {feature_names[feature_idx]} <= {threshold:.3f}? "
        f"value={value:.3f} -> {'left' if go_left else 'right'}"
    )
    node_id = tree_.children_left[node_id] if go_left else tree_.children_right[node_id]

leaf_class_idx = np.argmax(tree_.value[node_id])
print(f"Reached leaf node {node_id}; predicted class: {target_names[leaf_class_idx]}")


## Build Three Subset Trees

These small trees intentionally use different feature subsets. They are educational ensemble components, not a replacement for the selected model above.


In [ ]:
subsets = [
    [0, 1, 2],
    [0, 1, 3],
    [1, 2, 3],
]

all_preds = []
ensemble_rows = []
for indices in subsets:
    subset_name = ", ".join(feature_names[i] for i in indices)
    tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
    tree.fit(X_train[:, indices], y_train)
    preds = tree.predict(X_test[:, indices])
    all_preds.append(preds)
    ensemble_rows.append(
        {
            "model": f"DT({subset_name})",
            "accuracy": accuracy_score(y_test, preds),
            "macro_f1": f1_score(y_test, preds, average="macro"),
        }
    )


## Majority Vote Demonstration

A simple vote combines the three subset trees by taking the most common class prediction.


In [ ]:
vote_pred = mode(np.vstack(all_preds), axis=0, keepdims=False).mode
ensemble_rows.append(
    {
        "model": "Majority vote of three trees",
        "accuracy": accuracy_score(y_test, vote_pred),
        "macro_f1": f1_score(y_test, vote_pred, average="macro"),
    }
)


## Out-of-Fold Stacking Setup

The meta-model is trained on out-of-fold base probabilities, not on in-sample predictions. This avoids the classic stacking leakage mistake.


In [ ]:
def oof_probas_for_subset(indices):
    base = DecisionTreeClassifier(random_state=RANDOM_STATE)
    oof_proba = cross_val_predict(base, X_train[:, indices], y_train, cv=cv, method="predict_proba")
    base.fit(X_train[:, indices], y_train)
    test_proba = base.predict_proba(X_test[:, indices])
    return oof_proba, test_proba


oof_blocks = []
test_blocks = []
for indices in subsets:
    oof_proba, test_proba = oof_probas_for_subset(indices)
    oof_blocks.append(oof_proba)
    test_blocks.append(test_proba)

stack_train = np.hstack(oof_blocks)
stack_test = np.hstack(test_blocks)


## Train And Evaluate The Stacked Model

The stacker is a small logistic regression trained on the out-of-fold probability features.


In [ ]:
stacker = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
stacker.fit(stack_train, y_train)
stack_pred = stacker.predict(stack_test)
ensemble_rows.append(
    {
        "model": "OOF-stacked subset trees",
        "accuracy": accuracy_score(y_test, stack_pred),
        "macro_f1": f1_score(y_test, stack_pred, average="macro"),
    }
)

display(pd.DataFrame(ensemble_rows).sort_values("accuracy", ascending=False))


## Conclusion

The first half of the notebook is the primary model-selection workflow. The second half is an ensemble mechanics demonstration that explicitly uses out-of-fold predictions for stacking. Because the same small test split appears in multiple demos, those later results should be read as educational comparisons rather than a leaderboard.
